# Anisotropic stress and three-dimensional mixed spaces

Independent exact-field checks accompany five-level campaigns for anisotropic stress, MsHHO and interior/normal enrichment. These are formulation checks with declared meshes, rather than historical figure reproductions.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/6e14603e08c7cfd01cbd9590e60082b834a1617f4a8d74a89eaf815f493461d6/57_core_extensions-companion.zip"
COMPANION_SHA256 = "6e14603e08c7cfd01cbd9590e60082b834a1617f4a8d74a89eaf815f493461d6"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/general/57_core_extensions.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits


## Declared anisotropic stress equations

The primary path reuses the same explicit BDM2/P1/P1 block forms as the
mixed-stress introduction. The supplied full Cartesian compliance acts on
$(\sigma_{xx},\sigma_{xy},\sigma_{yx},\sigma_{yy})$ and retains the declared
positive skew extension. Shared kernels integrate that law; `LocalEquations`
explicitly includes physical displacement, independent weak rotation, local
boundary fields, three rigid modes and their moments. `Equation` and the
physical compliance-trace constraint declare the global displacement data and
hydrostatic condition. The displayed providers are the actual executed forms.

In [ ]:
import inspect
from IPython.display import Code
from pymhm.meshes.triangle import TriangleMesh
from examples import core_extension_data as data
from examples.tutorial_stress_equations import (
    solve_bdm, mixed_equations, bdm_equations, global_equation,
)

for builder in (mixed_equations, bdm_equations, global_equation):
    display(Code(inspect.getsource(builder), language="python"))

with threadpool_limits(1):
    mesh = TriangleMesh.unit_square(2)
    compliance = data.compliance()
    stress_system, stress = solve_bdm(
        mesh, compliance=compliance, source=data.force,
        dirichlet=data.displacement, local_refinement=1, quadrature_order=8,
    )
    print("Displacement L2 error:", stress.l2_error(data.displacement, 9))
    print("Full stress L2 error:", stress.stress_l2_error(data.stress, 9))

The compatibility solve below uses the identical anisotropic tensor, source,
nonhomogeneous trace, mesh and approximation spaces. Every mixed field,
trace and retained rigid coefficient is compared in its unchanged basis.

In [ ]:
from examples.formulations.application import weak_stress_elasticity as solve_elasticity_mixed

with threadpool_limits(1):
    stress_reference = solve_elasticity_mixed(
        mesh, compliance=compliance, source=data.force,
        dirichlet=data.displacement, local_refinement=1, quadrature_order=8,
    )
for own, reference in zip(
    (*stress.hybrid.fields, stress.hybrid.trace, *stress.hybrid.coarse),
    (*stress_reference.hybrid.fields, stress_reference.hybrid.trace,
     *stress_reference.hybrid.coarse), strict=True,
):
    np.testing.assert_allclose(own, reference, atol=1e-12, rtol=1e-10)
print("Anisotropic stress/displacement/rotation coefficients agree within atol=1e-12 and rtol=1e-10")

The mixed displacement is discontinuous. Profiles evaluate each fine-cell
polynomial independently; separate segments preserve one-sided values.
The profiles below use the archived n=4 solution, replayed with its stored
executed basis and orientation maps. The projection table covers n=2, 4, 8,
and 16. Its decomposition distinguishes approximation error from the
difference between the solved displacement and its best local approximation.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/core-extensions/elasticity-projection.json").read_text())
    for row in record["rows"]:
        print(row["resolution"], {name: float(np.linalg.norm(row[name])) for name in
              ("projection_l2", "discrete_l2", "total_l2")})
    display(Image(filename=str(ROOT / "docs/figures/core-extensions/polygon-bdm2-displacement-profiles.png")))


## Declared three-dimensional H(div) equations

The selected affine mixed family has pressure, normal and trace degrees two.
The provider declares its physical flux/pressure/normal-boundary saddle with
explicit Piola normal maps, the physical constant-pressure mode and its
volume moments. The same diagonal physical scaling transforms the operators,
loads, kernel and moments consistently. The global core assembles those actual
fields rather than choosing a solver by a formulation name.

The reference-element construction keeps all declared interior bubbles.
Orthogonal face tests and a weighted QR change of interior tests condition
the moment equations; the same change is applied to their prescribed right-hand
sides. Canonical normal moments and the positive-diagonal bubble orientation
remain the declared coordinates. Archived physical fields are evaluated with
the basis matrix that was actually executed.


In [ ]:
from pymhm.meshes.mixed import AffineMixedMesh
from pymhm.core.multiscale import assemble
from examples.formulations.mixed_darcy_3d import (
    define_hdiv_darcy, recover_hdiv_darcy,
    local_equations as hdiv_equations,
)
from examples.formulations.darcy import pressure_constraints as darcy_constraints

display(Code(inspect.getsource(hdiv_equations), language="python"))

def pressure(points: np.ndarray) -> np.ndarray:
    """Original quadratic exact pressure with represented normal flux traces."""
    return np.sum(points * points, axis=1)

mixed_mesh = AffineMixedMesh.unit_cube()
with threadpool_limits(1):
    mixed_definition = define_hdiv_darcy(
        mixed_mesh, pressure_degree=2, normal_degree=2, trace_degree=2,
        local_refinement=1, source=-6.0, dirichlet=pressure, quadrature_order=8,
    )
    mixed_system = assemble(mixed_definition.problem)
    mixed_coefficients = mixed_system.solve(
        constraints=darcy_constraints(mixed_definition, mixed_system),
    )
    mixed = recover_hdiv_darcy(mixed_definition, mixed_system, mixed_coefficients)
    assert mixed.errors(pressure, lambda x: -2 * x, 9)["flux_l2"] < 1e-10
    print(mixed.errors(pressure, lambda x: -2 * x, 9))

The earlier H(div) solver is evaluated separately on the same mesh and data.
This comparison includes the scaled saddle coefficients, trace and retained
constant-pressure amplitudes; physical pressure and Piola flux use their
executed coordinate transforms.

In [ ]:
from examples.formulations.application import hdiv_darcy as solve_darcy_hdiv3d

with threadpool_limits(1):
    mixed_reference = solve_darcy_hdiv3d(
        mixed_mesh, pressure_degree=2, normal_degree=2, trace_degree=2,
        local_refinement=1, source=-6.0, dirichlet=pressure, quadrature_order=8,
    )
for own, reference in zip(
    (*mixed.hybrid.fields, mixed.hybrid.trace, *mixed.hybrid.coarse),
    (*mixed_reference.hybrid.fields, mixed_reference.hybrid.trace,
     *mixed_reference.hybrid.coarse), strict=True,
):
    np.testing.assert_allclose(own, reference, atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(mixed.physical_residuals, mixed_reference.physical_residuals,
                           atol=1e-12, rtol=1e-10)
print("Original H(div) pressure/flux coefficients agree within atol=1e-12 and rtol=1e-10")

## Declared tetrahedral moment-energy equations

The constrained reconstruction uses the same original volume and unsigned
face functionals. The shared free `energy_reconstruction` constructs its actual
basis $R$ and represented energy $E$ without imposing an extra symmetry:

$$
\begin{aligned}
 A R+M\eta&=0, & M^\top R&=I, & E&=R^\top A R,\\
 E_{cc}u_c+E_{cf}u_f&=f_c,\\
 \sum_K(E_{fc}u_c+E_{ff}u_f)&=\sum_K f_f+b_N.
\end{aligned}
$$

`LocalEquations` explicitly declares these four cell/face blocks. The face
coordinates are pressure **integrals**. Physical outward Neumann flux supplies
the negative dual face load $b_N$. The volume mean includes both the recovered
cell and face contributions; the shared constraint/recovery functions are the
same ones used in the planar moment tutorial. This original control has one
constant cell moment, constant face moments and local P2 fields. Its recovered
$-A\nabla p$ is a raw flux; no H(div) conformity is assigned to that gradient.

In [ ]:
from pymhm.meshes.tetrahedron import TetraMesh
from examples.formulations.moments_3d import (
    define_moment_diffusion_3d, recover_moment_diffusion_3d,
    local_equations as moment_equations,
)
from examples.formulations.moments import pressure_constraints as moment_constraints

display(Code(inspect.getsource(moment_equations), language="python"))

hho_mesh = TetraMesh.unit_cube()
neumann = {
    int(face): -2 * hho_mesh.points[hho_mesh.faces[face]].mean(axis=0) @ hho_mesh.normals[face]
    for face in hho_mesh.boundary_faces
}
with threadpool_limits(1):
    moment_definition = define_moment_diffusion_3d(
        hho_mesh, source=-6.0, neumann=neumann, mean_pressure=1.0,
    )
    moment_system = assemble(moment_definition.problem)
    moment_coefficients = moment_system.solve(
        constraints=moment_constraints(moment_definition, moment_system),
    )
    hho = recover_moment_diffusion_3d(moment_definition, moment_system, moment_coefficients)
    assert hho.l2_error(pressure) < 2e-11
    assert hho.flux_l2_error(lambda x: -2 * x) < 2e-10
    print("MsHHO pressure / raw flux errors:",
          hho.l2_error(pressure), hho.flux_l2_error(lambda x: -2 * x))

The compatibility solve uses the identical constant source, physical Neumann
flux, volume mean, tetrahedral mesh and local/face spaces. Its constrained
reconstruction, moment matrix, energy, source functional and physical integral
are compared literally. Field differences from the common generic elimination
are checked separately at the scale of floating-point roundoff.

In [ ]:
from examples.formulations.application import moment_diffusion as solve_mshho_3d

with threadpool_limits(1):
    hho_reference = solve_mshho_3d(
        hho_mesh, source=-6.0, neumann=neumann, mean_pressure=1.0,
    )
for own, reference in zip(hho.local, hho_reference.local, strict=True):
    for name in ("reconstruction", "moments", "energy", "load", "integral"):
        np.testing.assert_allclose(getattr(own, name), getattr(reference, name),
                               atol=1e-12, rtol=1e-10)
maximum_difference = 0.0
for own, reference in zip(
    (*hho.pressure, hho.face_moments, *hho.cell_moments),
    (*hho_reference.pressure, hho_reference.face_moments,
     *hho_reference.cell_moments), strict=True,
):
    maximum_difference = max(maximum_difference, float(np.max(np.abs(own - reference))))
    np.testing.assert_allclose(own, reference, atol=2e-12, rtol=0)
print({"maximum_MsHHO_coefficient_difference": maximum_difference})

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for suite in ("elasticity", "hdiv3d", "mshho3d"):
        record = json.loads((ROOT / f"examples/results/core-extensions/{suite}.json").read_text())
        print(suite, len(record["rows"]), "acquired cases")
        display(Image(filename=str(ROOT / f"docs/figures/core-extensions/{suite}-convergence.png")))


The high-order comparison separates local interior degree and skeletal normal degree. Macro equilibrium and small algebraic residuals do not establish approximation accuracy. The records report physical norms and independently selected error quadratures.